In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 50.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import sklearn
print(sklearn.__version__)

1.6.1


In [3]:
!pip install git+https://github.com/cmu-phil/causal-learn.git

  Cloning https://github.com/cmu-phil/causal-learn.git to /tmp/pip-req-build-b448p6f4
  Running command git clone --filter=blob:none --quiet https://github.com/cmu-phil/causal-learn.git /tmp/pip-req-build-b448p6f4
  Resolved https://github.com/cmu-phil/causal-learn.git to commit 520728fa595641625946460a11f85e711d41a9e1
  Preparing metadata (setup.py) ... done
  Created wheel for causal-learn: filename=causal_learn-0.1.4.5-py3-none-any.whl size=255181 sha256=89cbec211126d7e1afd4f894c90ff788870392b815b870ffabfc2ef9f9da68c1
  Stored in directory: /tmp/pip-ephem-wheel-cache-45qnqnmg/wheels/a7/f5/19/0f01a36bb53da37cf83e9851609b81c542248a0373796c2361
Successfully built causal-learn


In [4]:
import os, gc, math, warnings, copy
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (accuracy_score, mutual_info_score,
                              balanced_accuracy_score, f1_score)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, demographic_parity_ratio

import joblib
from tqdm import tqdm

from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import chisq

warnings.filterwarnings('ignore')

SEED      = 42
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CACHE_DIR = './cache'
for _d in [CACHE_DIR, './results']:
    os.makedirs(_d, exist_ok=True)


def set_seed(s=SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()

PC_ALPHA        = {'adult': 0.05, 'compas': 0.05, 'german': 0.20, 'bank': 0.05,
                   'hospital': 0.05}
MI_SCREEN_ALPHA = {'adult': 0.15, 'compas': 0.15, 'german': 0.10, 'bank': 0.15,
                   'hospital': 0.15}
MI_LEAK_THRESH    = 0.40
MAX_MED_FEAT_FRAC = 0.40
MIN_MED_FRACTION  = 0.15

PRED_STD_MIN = {'adult': 0.05, 'compas': 0.04, 'german': 0.02,
                'bank': 0.02, 'hospital': 0.03}

FAIR_WEIGHTS = {
    'adult':    (6.0, 3.0, 1.0, 0.5, 1.0, 0.3),
    'compas':   (7.0, 3.5, 1.2, 0.5, 1.0, 0.3),
    'german':   (5.0, 2.0, 1.0, 0.5, 1.0, 0.3),
    'bank':     (4.0, 1.5, 0.8, 0.5, 1.0, 0.3),
    'hospital': (6.0, 3.0, 1.0, 0.5, 1.0, 0.3),
}

PARETO_LAMBDA = {
    'adult': 4.0, 'compas': 5.0, 'german': 3.5,
    'bank': 3.0, 'hospital': 4.0,
}

EPOCH_CFG = {
    'adult':    {'total': 1000, 'warmup': 30, 'patience': 120},
    'compas':   {'total': 1200, 'warmup': 50, 'patience': 150},
    'german':   {'total': 1200, 'warmup': 50, 'patience': 180},
    'bank':     {'total': 1000, 'warmup': 40, 'patience': 100},
    'hospital': {'total': 1000, 'warmup': 20, 'patience': 100},
}

RAMP_CENTRE = {'compas': 0.12, 'adult': 0.12, 'german': 0.15, 'bank': 0.15,
               'hospital': 0.10}
RAMP_STEEP  = {'compas': 16.0, 'adult': 16.0, 'german': 14.0, 'bank': 14.0,
               'hospital': 16.0}

COND_ALIGN_WEIGHT = {'adult': 0.5, 'compas': 0.5, 'german': 0.5,
                     'bank': 0.3, 'hospital': 0.5}

GATE_BIMODAL_WEIGHT = 0.3
MED_GRAD_SCALE      = 0.2

DATASET_CONFIG = {
    'adult':    {'sens_attrs': [('sens_sex_train',  'sens_sex_test'),
                                ('sens_race_train', 'sens_race_test')]},
    'compas':   {'sens_attrs': [('sens_race_train', 'sens_race_test'),
                                ('sens_sex_train',  'sens_sex_test')]},
    'german':   {'sens_attrs': [('sens_age_train',  'sens_age_test'),
                                ('sens_sex_train',  'sens_sex_test')]},
    'bank':     {'sens_attrs': [('sens_marital_train', 'sens_marital_test'),
                                ('sens_job_train',     'sens_job_test')]},
    'hospital': {'sens_attrs': [('sens_race_train', 'sens_race_test'),
                                ('sens_sex_train',  'sens_sex_test')]},
}

LABEL_LEAK = {
    'readmit_binary', 'readmitted', 'two_year_recid', 'is_recid',
    'decile_score', 'score_text', 'bar', 'bar2', 'bar_passed', 'bar_pass',
    'zgpa', 'zfygpa', 'indxgrp', 'indxgrp2', 'decile1b', 'decile3',
    'dnn_bar_pass_prediction', 'decile1', 'index6040', 'bar2_yr', 'cluster',
    'bar1', 'bar_pass_prediction',
    'lsat', 'ugpa', 'gpa',
}

ARTIFACT_COLS = {
    'id', '_id', 'index', 'row', 'sample', 'name', 'last', 'first', 'dob',
    'compas_screening_date', 'c_case_number', 'r_case_number', 'vr_case_number',
    'r_offense_date', 'vr_offense_date', 'c_arrest_date', 'c_offense_date',
    'c_jail_in', 'c_jail_out', 'v_screening_date', 'in_custody', 'out_custody',
    'event', 'type_of_assessment', 'v_type_of_assessment', 'score_factor',
    'r_charge_desc', 'vr_charge_desc', 'r_charge_degree', 'vr_charge_degree',
    'screening_date', 'race1', 'race2', 'race_bin', 'sex_bin', 'gender_bin',
    'age_bin', 'marital_bin', 'job_bin', 'black', 'white', 'asian', 'hispanic',
}


def to_dense(X):
    arr = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


def clean_num(s):
    s = pd.to_numeric(s, errors='coerce')
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)


def safe_qcut(s, q=5):
    s = clean_num(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        return pd.Series(0, index=s.index, dtype=int)


def num_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler())])


def cat_pipe():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore',
                                           sparse_output=False))])


def _safe_cat_codes(s):
    if pd.api.types.is_categorical_dtype(s):
        return s.cat.codes.astype(int)
    return s.astype('category').cat.codes.astype(int)


def _enc_objects(df):
    for col in df.select_dtypes(include=['object', 'category']).columns:
        df[col] = df[col].astype('category').cat.codes.astype(int)
    return df


def _filter_bbn(df, sens_col, y_col, extra_drop=()):
    drop = set(extra_drop)
    n = len(df)
    for c in df.columns:
        if c in (sens_col, y_col):
            continue
        cl = c.lower().strip()
        if cl in LABEL_LEAK or cl in ARTIFACT_COLS:
            drop.add(c)
            continue
        if df[c].nunique() <= 1:
            drop.add(c)
            continue
        if df[c].nunique() > min(50, max(10, int(n * 0.05))):
            drop.add(c)
    return df.drop(columns=[c for c in drop if c in df.columns])


def _nn_feature_names(pre, num_c, cat_c):
    names = list(num_c)
    ohe = pre.named_transformers_['c'].named_steps['ohe']
    for i, c in enumerate(cat_c):
        for v in ohe.categories_[i]:
            names.append(f'{c}={v}')
    return names


def load_adult(csv_path='/kaggle/input/datasets/maansitemp/all-datasets/adult.csv',
               seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'adult_v27.pkl')
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    cols = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
            'marital-status', 'occupation', 'relationship', 'race', 'sex',
            'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
    df = pd.read_csv(csv_path, skipinitialspace=True, header=None, names=cols)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df = df.drop(columns=['fnlwgt'])
    df['y']        = df['income'].str.contains('>50K', na=False).astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['race_bin'] = (df['race'] == 'White').astype(int)
    num_c = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
    cat_c = ['workclass', 'education', 'marital-status', 'occupation',
             'relationship', 'native-country']
    for c in num_c:
        df[c] = clean_num(df[c])
    for c in ['capital-gain', 'capital-loss']:
        df[c] = df[c].clip(upper=df[c].quantile(0.99))
    X = df.drop(columns=['income', 'y', 'sex_bin', 'race_bin', 'sex', 'race'])
    y = df['y'].values
    (X_tr, X_te, y_tr, y_te,
     ss_tr, ss_te, sr_tr, sr_te) = train_test_split(
        X, y, df['sex_bin'].values, df['race_bin'].values,
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', num_pipe(), num_c), ('c', cat_pipe(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    nn_names = _nn_feature_names(pre, num_c, cat_c)
    bbn = df[[c for c in num_c + cat_c + ['y', 'sex_bin', 'race_bin']
              if c in df.columns]].copy()
    for c in num_c:
        if c in bbn: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c:
        if c in bbn: bbn[c] = _safe_cat_codes(bbn[c])
    bbn['y'] = bbn['y'].astype(int)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _filter_bbn(bbn_tr, 'sex_bin', 'y', ('sex', 'race', 'race_bin', 'sex_bin'))
    bbn_te = _filter_bbn(bbn_te, 'sex_bin', 'y', ('sex', 'race', 'race_bin', 'sex_bin'))
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             sens_race_train=sr_tr, sens_race_test=sr_te,
             num_cols=num_c, cat_cols=cat_c, nn_feature_names=nn_names)
    if use_cache: joblib.dump(r, cache)
    return r


def load_compas(csv_path='/kaggle/input/datasets/maansitemp/all-datasets/compas-scores-two-years.csv',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'compas_v27.pkl')
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'].between(-30, 30)) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A')
    ].reset_index(drop=True)
    df['race_bin'] = (df['race'] == 'African-American').astype(int)
    df['sex_bin']  = (df['sex']  == 'Male').astype(int)
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'],  errors='coerce')
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'], errors='coerce')
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in', 'c_jail_out'])
    num_c = ['age', 'priors_count', 'days_b_screening_arrest', 'jail_time',
             'juv_other_count', 'juv_misd_count', 'juv_fel_count']
    cat_c = ['c_charge_degree', 'race', 'age_cat', 'sex', 'c_charge_desc']
    keep  = [c for c in num_c + cat_c + ['two_year_recid', 'race_bin', 'sex_bin']
             if c in df.columns]
    df = df[keep].copy()
    for c in num_c:
        if c in df: df[c] = clean_num(df[c])
    X = df.drop(columns=[c for c in ['two_year_recid', 'race_bin', 'sex_bin']
                          if c in df.columns])
    y = df['two_year_recid'].values
    (X_tr, X_te, y_tr, y_te,
     sr_tr, sr_te, ss_tr, ss_te) = train_test_split(
        X, y, df['race_bin'], df['sex_bin'],
        test_size=0.2, stratify=y, random_state=seed)
    act_num = [c for c in num_c if c in X.columns]
    act_cat = [c for c in cat_c if c in X.columns]
    pre = ColumnTransformer([('n', num_pipe(), act_num), ('c', cat_pipe(), act_cat)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    nn_names = _nn_feature_names(pre, act_num, act_cat)
    bbn = df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c + ['race', 'sex']:
        if c in bbn: bbn[c] = _safe_cat_codes(bbn[c])
    bbn = _enc_objects(bbn)
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _filter_bbn(bbn_tr, 'race_bin', 'two_year_recid',
                         ('race', 'sex', 'sex_bin', 'race_bin', 'is_recid'))
    bbn_te = _filter_bbn(bbn_te, 'race_bin', 'two_year_recid',
                         ('race', 'sex', 'sex_bin', 'race_bin', 'is_recid'))
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr, sens_race_test=sr_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             num_cols=act_num, cat_cols=act_cat, nn_feature_names=nn_names)
    if use_cache: joblib.dump(r, cache)
    return r


def load_german(csv_path='/kaggle/input/datasets/maansitemp/all-datasets/german.data',
                seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'german_v28.pkl')
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    cols = ['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings',
            'employment', 'installment_rate', 'personal_status_sex', 'other_debtors',
            'residence', 'property', 'age', 'other_installment_plans', 'housing',
            'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker', 'credit']
    df = pd.read_csv(csv_path, sep=' ', names=cols)
    sex_map = {'A91': 'male', 'A92': 'male', 'A93': 'male',
               'A94': 'female', 'A95': 'female'}
    df['sex']     = df['personal_status_sex'].map(sex_map)
    df['sex_bin'] = (df['sex'] == 'male').astype(int)
    df['age_bin'] = (df['age'] >= 25).astype(int)
    df['y']       = df['credit'].map({1: 1, 2: 0})
    df = df.drop(columns=['personal_status_sex', 'sex', 'age', 'foreign_worker', 'credit'])
    num_c = ['duration', 'amount', 'installment_rate', 'residence',
             'number_credits', 'people_liable']
    cat_c = [c for c in df.columns if c not in num_c + ['sex_bin', 'age_bin', 'y']]
    for c in num_c:
        df[c] = clean_num(df[c])
    X = df.drop(columns=['y'])
    y = df['y'].values
    strat_key = df['y'].astype(str) + '_' + df['age_bin'].astype(str)
    (X_tr, X_te, y_tr, y_te,
     sa_tr, sa_te, ss_tr, ss_te) = train_test_split(
        X, y, df['age_bin'].values, df['sex_bin'].values,
        test_size=0.2, stratify=strat_key, random_state=seed)
    pre = ColumnTransformer([('n', num_pipe(), num_c), ('c', cat_pipe(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    nn_names = _nn_feature_names(pre, num_c, cat_c)
    keep = [c for c in num_c + cat_c + ['y', 'sex_bin', 'age_bin'] if c in df.columns]
    bbn  = df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c:
        if c in bbn: bbn[c] = _safe_cat_codes(bbn[c])
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _filter_bbn(bbn_tr, 'age_bin', 'y', ('sex_bin', 'age_bin'))
    bbn_te = _filter_bbn(bbn_te, 'age_bin', 'y', ('sex_bin', 'age_bin'))
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_age_train=sa_tr, sens_age_test=sa_te,
             sens_sex_train=ss_tr, sens_sex_test=ss_te,
             num_cols=num_c, cat_cols=cat_c, nn_feature_names=nn_names)
    if use_cache: joblib.dump(r, cache)
    return r


def load_bank(csv_path='/kaggle/input/datasets/maansitemp/all-datasets/bank-full.csv',
              seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'bank_v27.pkl')
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path, sep=';')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = (df['y'] == 'yes').astype(int)
    df['marital_bin'] = (df['marital'] ==
                         df['marital'].value_counts().idxmax()).astype(int)
    df['job_bin']     = (df['job']     ==
                         df['job'].value_counts().idxmax()).astype(int)
    cat_c = [c for c in ['job', 'education', 'default', 'housing', 'loan',
                          'contact', 'month', 'poutcome'] if c in df.columns]
    num_c = [c for c in ['age', 'balance', 'day', 'duration', 'campaign',
                          'pdays', 'previous'] if c in df.columns]
    for c in num_c:
        df[c] = clean_num(df[c])
    X = df.drop(columns=['y', 'marital_bin', 'job_bin', 'marital'])
    y = df['y'].values
    (X_tr, X_te, y_tr, y_te,
     sm_tr, sm_te, sj_tr, sj_te) = train_test_split(
        X, y, df['marital_bin'], df['job_bin'],
        test_size=0.2, stratify=y, random_state=seed)
    cat_enc = [c for c in cat_c if c in X.columns]
    num_enc = [c for c in num_c if c in X.columns]
    pre = ColumnTransformer([('n', num_pipe(), num_enc), ('c', cat_pipe(), cat_enc)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    nn_names = _nn_feature_names(pre, num_enc, cat_enc)
    keep = [c for c in num_c + cat_c + ['marital', 'y', 'marital_bin', 'job_bin']
            if c in df.columns]
    bbn  = df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c + ['marital']:
        if c in bbn: bbn[c] = _safe_cat_codes(bbn[c])
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _filter_bbn(bbn_tr, 'marital_bin', 'y',
                         ('marital', 'job_bin', 'marital_bin'))
    bbn_te = _filter_bbn(bbn_te, 'marital_bin', 'y',
                         ('marital', 'job_bin', 'marital_bin'))
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_marital_train=sm_tr, sens_marital_test=sm_te,
             sens_job_train=sj_tr, sens_job_test=sj_te,
             num_cols=num_enc, cat_cols=cat_enc, nn_feature_names=nn_names)
    if use_cache: joblib.dump(r, cache)
    return r


def load_hospital(csv_path='/kaggle/input/datasets/maansitemp/all-datasets/diabetes_hospital_fairlearn.csv',
                  seed=SEED, use_cache=True):
    cache = os.path.join(CACHE_DIR, 'hospital_v27.pkl')
    if use_cache and os.path.exists(cache):
        return joblib.load(cache)
    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ['max_glu_serum', 'A1Cresult', 'readmitted.1']
                           if c in df.columns])
    df = (df[~df.isin(['Missing']).any(axis=1)]
            .dropna(subset=['race', 'gender'])
            .reset_index(drop=True))
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20, "'30-60 years'": 45, "'Over 60 years'": 70,
    }
    df['age'] = df['age'].replace(age_map).astype(float)
    df['y']   = df['readmit_binary'].astype(int)
    mc = df['race'].value_counts().idxmax()
    df['race_bin']   = (df['race']   == mc).astype(int)
    df['gender_bin'] = df['gender'].map({'Male': 0, 'Female': 1}).fillna(0).astype(int)
    cat_c = [c for c in ['discharge_disposition_id', 'admission_source_id',
                          'medical_specialty', 'primary_diagnosis',
                          'insulin', 'change', 'diabetesMed'] if c in df.columns]
    num_c = [c for c in ['age', 'time_in_hospital', 'num_lab_procedures',
                          'num_procedures', 'num_medications', 'number_diagnoses',
                          'had_emergency', 'had_inpatient_days', 'had_outpatient_days',
                          'medicare', 'medicaid'] if c in df.columns]
    for c in num_c:
        df[c] = clean_num(df[c])
    X = df.drop(columns=['readmit_binary', 'y', 'race_bin', 'gender_bin'])
    y = df['y'].values
    (X_tr, X_te, y_tr, y_te,
     sr_tr, sr_te, sg_tr, sg_te) = train_test_split(
        X, y, df['race_bin'], df['gender_bin'],
        test_size=0.2, stratify=y, random_state=seed)
    pre = ColumnTransformer([('n', num_pipe(), num_c), ('c', cat_pipe(), cat_c)])
    X_tr_nn = to_dense(pre.fit_transform(X_tr))
    X_te_nn = to_dense(pre.transform(X_te))
    nn_names = _nn_feature_names(pre, num_c, cat_c)
    keep = [c for c in num_c + cat_c + ['y', 'race_bin', 'gender_bin', 'race', 'gender']
            if c in df.columns]
    bbn  = df[keep].copy()
    for c in num_c:
        if c in bbn: bbn[c] = safe_qcut(bbn[c], 5)
    for c in cat_c + ['race', 'gender']:
        if c in bbn: bbn[c] = _safe_cat_codes(bbn[c])
    bbn_tr = bbn.loc[X_tr.index].reset_index(drop=True)
    bbn_te = bbn.loc[X_te.index].reset_index(drop=True)
    bbn_tr = _filter_bbn(bbn_tr, 'race_bin', 'y',
                         ('race', 'gender', 'race_bin', 'gender_bin', 'readmit_binary'))
    bbn_te = _filter_bbn(bbn_te, 'race_bin', 'y',
                         ('race', 'gender', 'race_bin', 'gender_bin', 'readmit_binary'))
    r = dict(X_train_nn=X_tr_nn, X_test_nn=X_te_nn,
             bbn_train_df=bbn_tr, bbn_test_df=bbn_te,
             y_train=y_tr, y_test=y_te,
             sens_race_train=sr_tr, sens_race_test=sr_te,
             sens_sex_train=sg_tr, sens_sex_test=sg_te,
             num_cols=num_c, cat_cols=cat_c, nn_feature_names=nn_names)
    if use_cache: joblib.dump(r, cache)
    return r


def _mi_screen_mediators(bbn_df, sens_col, label_col, nn_feature_names,
                          alpha_mi=0.15, leak_thresh=MI_LEAK_THRESH,
                          max_feat_frac=MAX_MED_FEAT_FRAC):
    df = bbn_df.copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
    if sens_col not in df.columns or label_col not in df.columns:
        return [], []
    s     = df[sens_col].values
    y     = df[label_col].values
    mi_sy = mutual_info_score(s, y)
    h_y   = mutual_info_score(y, y)
    if mi_sy < 1e-8:
        tqdm.write('  [MI-screen] MI(S,Y) near zero — skipping')
        return [], []
    tqdm.write(f'  [MI-screen] MI(S,Y)={mi_sy:.4f}  H(Y)={h_y:.4f}  '
               f'thresh={alpha_mi * mi_sy:.4f}')
    scored = []
    for col in df.columns:
        if col in (sens_col, label_col):
            continue
        cl = col.lower().strip()
        if cl in LABEL_LEAK or cl in ARTIFACT_COLS:
            continue
        f     = df[col].values
        mi_sf = mutual_info_score(s, f)
        mi_fy = mutual_info_score(f, y)
        if mi_fy / (h_y + 1e-8) > leak_thresh:
            tqdm.write(f'  [MI-screen] SKIP {col}: leak={mi_fy / (h_y + 1e-8):.3f}')
            continue
        if mi_sf > alpha_mi * mi_sy and mi_fy > alpha_mi * mi_sy:
            scored.append((col, mi_sf * mi_fy))
    scored.sort(key=lambda x: -x[1])
    max_bbn = max(1, int(max_feat_frac * (len(df.columns) - 2)))
    scored  = scored[:max_bbn]
    med_bbn_names  = [c for c, _ in scored]
    med_nn_indices = []
    for m_name in med_bbn_names:
        for i, fn in enumerate(nn_feature_names):
            if fn == m_name or fn.startswith(m_name + '='):
                if i not in med_nn_indices:
                    med_nn_indices.append(i)
    return med_bbn_names, med_nn_indices


def _hsic_sigma(z1, z2):
    with torch.no_grad():
        d1 = torch.cdist(z1.detach(), z1.detach())
        d2 = torch.cdist(z2.detach(), z2.detach())
        s1 = d1[d1 > 0].median().item() if (d1 > 0).any() else 1.0
        s2 = d2[d2 > 0].median().item() if (d2 > 0).any() else 1.0
    return math.sqrt(s1 * s2) / math.sqrt(2.0)


def _hsic(z1, z2):
    n = z1.shape[0]
    if n < 4:
        return torch.tensor(0.0, device=z1.device)
    sigma = max(_hsic_sigma(z1, z2), 0.1)

    def rbf(x):
        sq = torch.cdist(x, x).pow(2)
        return torch.exp(-sq / (2.0 * sigma ** 2))

    K  = rbf(z1)
    L  = rbf(z2)
    H  = torch.eye(n, device=z1.device) - (1.0 / n)
    KH = K @ H
    return (KH * (H @ L).T).sum() / ((n - 1) ** 2)


def learn_causal_mediators(bbn_df, sens_col, label_col, nn_feature_names,
                            alpha=0.05, mi_alpha=0.15, dataset_name='',
                            leak_thresh=MI_LEAK_THRESH,
                            max_feat_frac=MAX_MED_FEAT_FRAC):
    df = bbn_df.copy()
    df = _enc_objects(df)
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
    keep_cols = [c for c in df.columns if df[c].nunique() > 1]
    df = df[keep_cols]
    if sens_col not in df.columns or label_col not in df.columns:
        return [], [], {}

    col_order = ([sens_col] +
                 [c for c in df.columns if c not in (sens_col, label_col)] +
                 [label_col])
    df       = df[col_order]
    data_mat = df.values.astype(float)
    s_idx    = 0
    y_idx    = len(col_order) - 1
    n_nodes  = data_mat.shape[1]
    tqdm.write(f'  [PC] {dataset_name}: {n_nodes} nodes, n={len(df)}, alpha={alpha}')

    pc_med_bbn = []
    dag_info   = {}
    try:
        cg  = pc(data_mat, alpha=alpha, indep_test=chisq, show_progress=False)
        G   = cg.G.graph
        adj = {i: set() for i in range(n_nodes)}
        for i in range(n_nodes):
            for j in range(n_nodes):
                if G[i, j] != 0:
                    adj[i].add(j)

        directed  = {i: set() for i in range(n_nodes)}
        processed = set()
        for i in range(n_nodes):
            for j in adj[i]:
                pair = tuple(sorted([i, j]))
                if pair in processed:
                    continue
                processed.add(pair)
                a, b = pair
                if s_idx in (a, b):
                    directed[s_idx].add(b if a == s_idx else a)
                elif y_idx in (a, b):
                    directed[a if b == y_idx else b].add(y_idx)
                else:
                    if   G[b, a] == 1 and G[a, b] == -1: directed[b].add(a)
                    elif G[a, b] == 1 and G[b, a] == -1: directed[a].add(b)
                    else: directed[min(a, b)].add(max(a, b))

        def reach(start, g):
            visited, stack = set(), [start]
            while stack:
                node = stack.pop()
                if node not in visited:
                    visited.add(node)
                    stack.extend(g[node] - visited)
            return visited

        from_s = reach(s_idx, directed)
        rev    = {i: set() for i in range(n_nodes)}
        for i, js in directed.items():
            for j in js: rev[j].add(i)
        to_y = reach(y_idx, rev)
        med_indices_bbn = (from_s & to_y) - {s_idx, y_idx}

        s_vals = data_mat[:, s_idx]
        y_vals = data_mat[:, y_idx]
        h_y    = mutual_info_score(y_vals, y_vals)
        scored_pc = []
        for m in med_indices_bbn:
            f_vals = data_mat[:, m]
            mi_sf  = mutual_info_score(s_vals, f_vals)
            mi_fy  = mutual_info_score(f_vals, y_vals)
            if mi_fy / (h_y + 1e-8) > leak_thresh:
                tqdm.write(f'  [PC leakage guard] SKIP {col_order[m]}')
            else:
                scored_pc.append((m, mi_sf * mi_fy))
        scored_pc.sort(key=lambda x: -x[1])
        max_pc = max(1, int(max_feat_frac * (n_nodes - 2)))
        scored_pc = scored_pc[:max_pc]
        clean_med_indices = {m for m, _ in scored_pc}
        pc_med_bbn = [col_order[m] for m in sorted(clean_med_indices)]

        dag_info = {
            'col_order': col_order,
            'directed_edges': {
                col_order[i]: [col_order[j] for j in js]
                for i, js in directed.items()
            },
            'mediator_bbn_names': pc_med_bbn,
            'from_s': [col_order[i] for i in from_s if i not in (s_idx, y_idx)],
            'to_y':   [col_order[i] for i in to_y   if i not in (s_idx, y_idx)],
            'n_nodes': n_nodes, 's_idx': s_idx, 'y_idx': y_idx,
            'med_indices_bbn': sorted(clean_med_indices),
        }
        tqdm.write(f'  [PC] Mediators (capped+leak guard): {pc_med_bbn}')
    except Exception as e:
        tqdm.write(f'  [PC] Failed: {e}')

    mi_med_bbn, mi_med_nn = _mi_screen_mediators(
        bbn_df, sens_col, label_col, nn_feature_names,
        alpha_mi=mi_alpha, leak_thresh=leak_thresh,
        max_feat_frac=max_feat_frac,
    )
    tqdm.write(f'  [MI-screen] Candidates: {mi_med_bbn}')

    all_med_bbn = list(dict.fromkeys(
        pc_med_bbn + [m for m in mi_med_bbn if m not in pc_med_bbn]))
    n_bbn_feats = len(bbn_df.columns) - 2
    hard_cap    = max(1, int(max_feat_frac * n_bbn_feats))
    if len(all_med_bbn) > hard_cap:
        s_col = bbn_df[sens_col].fillna(0).astype(int) if sens_col in bbn_df.columns \
            else pd.Series(0, index=bbn_df.index)
        y_col_s = bbn_df[label_col].fillna(0).astype(int) if label_col in bbn_df.columns \
            else pd.Series(0, index=bbn_df.index)
        scored_union = []
        for name in all_med_bbn:
            if name not in bbn_df.columns: continue
            f = bbn_df[name].fillna(0).astype(int)
            scored_union.append(
                (name, mutual_info_score(s_col, f) * mutual_info_score(f, y_col_s)))
        scored_union.sort(key=lambda x: -x[1])
        all_med_bbn = [c for c, _ in scored_union[:hard_cap]]
        tqdm.write(f'  [CAP] Trimmed to {hard_cap} mediators by MI product score')

    all_med_nn = []
    for m_name in all_med_bbn:
        for i, fn in enumerate(nn_feature_names):
            if fn == m_name or fn.startswith(m_name + '='):
                if i not in all_med_nn:
                    all_med_nn.append(i)

    dag_info['mediator_bbn_names'] = all_med_bbn
    dag_info['pc_only_mediators']  = [m for m in pc_med_bbn if m not in mi_med_bbn]
    dag_info['mi_only_mediators']  = [m for m in mi_med_bbn if m not in pc_med_bbn]
    dag_info['both_mediators']     = [m for m in pc_med_bbn if m in mi_med_bbn]

    tqdm.write(
        f'  [Hybrid] Final mediators ({len(all_med_bbn)}): {all_med_bbn}\n'
        f'    PC-only: {dag_info["pc_only_mediators"]}\n'
        f'    MI-only: {dag_info["mi_only_mediators"]}\n'
        f'    Both:    {dag_info["both_mediators"]}\n'
        f'    NN idx:  {all_med_nn}/{len(nn_feature_names)}'
    )
    return all_med_bbn, all_med_nn, dag_info


def build_dag_mask_init(dag_info, nn_feature_names, z_dim=64):
    min_med_dims = max(1, int(MIN_MED_FRACTION * z_dim))
    if not dag_info or 'col_order' not in dag_info:
        mask_vals = np.zeros(z_dim)
        mask_vals[:min_med_dims] = 0.7
        logit_mask = np.log(
            np.clip(mask_vals, 1e-6, 1 - 1e-6) /
            (1 - np.clip(mask_vals, 1e-6, 1 - 1e-6)))
        return torch.tensor(logit_mask, dtype=torch.float32)

    col_order      = dag_info['col_order']
    directed_edges = dag_info.get('directed_edges', {})
    med_bbn_names  = set(dag_info['mediator_bbn_names'])
    from_s_names   = set(dag_info.get('from_s', []))
    to_y_names     = set(dag_info.get('to_y', []))

    node_score = {}
    for name in col_order:
        if   name in med_bbn_names:                        node_score[name] = 1.0
        elif name in from_s_names and name in to_y_names: node_score[name] = 1.0
        elif name in from_s_names:                         node_score[name] = 0.6
        elif name in to_y_names:                           node_score[name] = 0.2
        else:                                              node_score[name] = 0.0

    propagated = dict(node_score)
    for src, dsts in directed_edges.items():
        src_score = node_score.get(src, 0.0)
        for dst in dsts:
            propagated[dst] = max(propagated.get(dst, 0.0), src_score * 0.5)
    for name in col_order:
        propagated[name] = max(node_score.get(name, 0.0), propagated.get(name, 0.0))

    feat_scores = np.zeros(len(nn_feature_names))
    for i, fn in enumerate(nn_feature_names):
        base = fn.split('=')[0]
        feat_scores[i] = propagated.get(base, propagated.get(fn, 0.0))

    sorted_scores = np.sort(feat_scores)[::-1]
    chunk = max(1, len(sorted_scores) // z_dim)
    mask_vals = np.zeros(z_dim)
    for d in range(z_dim):
        start = d * chunk
        end   = min(start + chunk, len(sorted_scores))
        mask_vals[d] = float(np.mean(sorted_scores[start:end])) \
            if start < end else 0.0

    high_dims = int((mask_vals > 0.5).sum())
    if high_dims < min_med_dims:
        top_k_idx = np.argsort(mask_vals)[::-1][:min_med_dims]
        for idx in top_k_idx:
            mask_vals[idx] = max(mask_vals[idx], 0.70)
        tqdm.write(f'  [DAG-mask] Boosted {min_med_dims - high_dims} dims to '
                   f'meet min_med_fraction={MIN_MED_FRACTION}')

    eps       = 1e-6
    mask_vals = np.clip(mask_vals, eps, 1 - eps)
    logit_mask = np.log(mask_vals / (1 - mask_vals))
    med_count  = int((mask_vals > 0.5).sum())
    tqdm.write(f'  [DAG-mask] med_score>0.5 dims: {med_count}/{z_dim}')
    tqdm.write(f'  [DAG-mask] score dist: max={mask_vals.max():.3f} '
               f'mean={mask_vals.mean():.3f} min={mask_vals.min():.3f}')
    return torch.tensor(logit_mask, dtype=torch.float32)


class SharedTrunk(nn.Module):
    def __init__(self, input_dim, trunk_dim=128, dropout=0.3):
        super().__init__()
        h = 256
        self.net = nn.Sequential(
            nn.Linear(input_dim, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, h),         nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, trunk_dim), nn.LayerNorm(trunk_dim),
        )

    def forward(self, x):
        return self.net(x)


class InputConditionedGate(nn.Module):
    def __init__(self, trunk_dim, z_dim, init_logit_bias=None):
        super().__init__()
        hidden = max(32, z_dim // 2)
        self.net = nn.Sequential(
            nn.Linear(trunk_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, z_dim),
        )
        if init_logit_bias is not None:
            with torch.no_grad():
                self.net[-1].bias.copy_(init_logit_bias)

    def forward(self, h):
        return torch.sigmoid(self.net(h))


class CausalSplitHead(nn.Module):
    def __init__(self, trunk_dim, z_dim, mask_init):
        super().__init__()
        self.z_dim     = z_dim
        self.proj_task = nn.Linear(trunk_dim, z_dim)
        self.proj_med  = nn.Linear(trunk_dim, z_dim)
        self.gate_net  = InputConditionedGate(trunk_dim, z_dim,
                                               init_logit_bias=mask_init)
        self.norm_task = nn.LayerNorm(z_dim)
        self.norm_med  = nn.LayerNorm(z_dim)

    def forward(self, h):
        gate   = self.gate_net(h)
        z_task = self.norm_task(self.proj_task(h) * (1.0 - gate))
        z_med  = self.norm_med (self.proj_med(h)  * gate)
        return z_task, z_med, gate

    def gate_regularisation_loss(self, gate):
        gate_mean   = gate.mean(dim=0)
        centre_pen  = torch.abs(gate_mean.mean() - 0.5)
        bimodal_pen = (gate * (1.0 - gate)).mean()
        return GATE_BIMODAL_WEIGHT * bimodal_pen + 0.1 * centre_pen

    @torch.no_grad()
    def get_gate_stats(self, h_sample):
        g = self.gate_net(h_sample).cpu().numpy()
        return float(g.mean()), float(g.std()), int((g.mean(axis=0) > 0.5).sum())


class TaskHead(nn.Module):
    def __init__(self, z_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 64), nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, z_task):
        return self.net(z_task).squeeze(-1)


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class SensAdv(nn.Module):
    def __init__(self, z_dim=64, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 2),
        )

    def forward(self, z, alpha=1.0):
        return self.net(GradReverse.apply(z, alpha))


class SensProbe(nn.Module):
    def __init__(self, z_dim=64, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.GELU(),
            nn.Linear(hidden, 2)
        )

    def forward(self, z):
        return self.net(z)


class MedDecoder(nn.Module):
    def __init__(self, z_dim=64, med_dim=1):
        super().__init__()
        h = max(32, med_dim)
        self.net = nn.Sequential(
            nn.Linear(z_dim, h), nn.GELU(),
            nn.Linear(h, med_dim)
        )

    def forward(self, z):
        return self.net(z)


def fair_loss_dp_soft(logit, s, margin=0.0):
    p  = torch.sigmoid(logit)
    p0 = p[s == 0].mean() if (s == 0).sum() > 2 \
        else torch.tensor(0.5, device=logit.device)
    p1 = p[s == 1].mean() if (s == 1).sum() > 2 \
        else torch.tensor(0.5, device=logit.device)
    return F.relu(torch.abs(p0 - p1) - margin)


@torch.no_grad()
def get_proba(trunk, split_head, task_head, X_all, device):
    trunk.eval(); split_head.eval(); task_head.eval()
    X_t = torch.tensor(X_all, dtype=torch.float32).to(device)
    h = trunk(X_t)
    z_task, _, _ = split_head(h)
    return torch.sigmoid(task_head(z_task)).cpu().numpy()


class ParetoCheckpoint:
    def __init__(self, lam=3.0):
        self.lam            = lam
        self.best_composite = -float('inf')
        self.best_state     = None
        self.best_acc       = 0.0
        self.best_metric    = 1.0

    def update(self, acc, metric, **modules):
        composite = acc - self.lam * metric
        if composite > self.best_composite:
            self.best_composite = composite
            self.best_acc       = acc
            self.best_metric    = metric
            self.best_state = {
                k: copy.deepcopy(v.state_dict()) if v is not None else None
                for k, v in modules.items()
            }
            return True
        return False

    def restore(self, trunk, split_head, task_head, sens_probe=None):
        if self.best_state is None:
            return
        trunk.load_state_dict(self.best_state['trunk'])
        split_head.load_state_dict(self.best_state['split_head'])
        task_head.load_state_dict(self.best_state['task_head'])
        if sens_probe is not None and self.best_state.get('sens_probe'):
            sens_probe.load_state_dict(self.best_state['sens_probe'])


class EarlyStopping:
    def __init__(self, patience=80, min_delta=1e-4):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_val    = float('inf')
        self.no_improve  = 0

    def update(self, composite_loss):
        if composite_loss < self.best_val - self.min_delta:
            self.best_val   = composite_loss
            self.no_improve = 0
        else:
            self.no_improve += 1
        return self.no_improve >= self.patience


def train(dataset_name, data, med_nn_idx, dag_info, s_train, s_test,
          use_adv=True, use_fair=True, use_dag_mask=True):
    cfg        = EPOCH_CFG.get(dataset_name, {'total': 1000, 'warmup': 10, 'patience': 120})
    total_ep   = cfg['total']
    warmup_ep  = cfg['warmup'] if use_fair else total_ep + 1
    patience   = cfg['patience']
    batch_size = 2048

    X_nn       = data['X_train_nn']
    y_np       = data['y_train']
    n_features = X_nn.shape[1]
    nn_names   = data['nn_feature_names']
    has_med    = len(med_nn_idx) > 0
    z_dim      = 64
    trunk_dim  = 128

    lam = PARETO_LAMBDA.get(dataset_name, 3.0)

    if use_dag_mask:
        mask_init = build_dag_mask_init(dag_info, nn_names, z_dim=z_dim).to(DEVICE)
    else:
        mask_init = torch.zeros(z_dim).to(DEVICE)

    trunk      = SharedTrunk(input_dim=n_features, trunk_dim=trunk_dim).to(DEVICE)
    split_head = CausalSplitHead(trunk_dim=trunk_dim, z_dim=z_dim,
                                  mask_init=mask_init).to(DEVICE)
    task_head  = TaskHead(z_dim=z_dim).to(DEVICE)
    sens_adv   = SensAdv(z_dim=z_dim).to(DEVICE)
    sens_probe = SensProbe(z_dim=z_dim).to(DEVICE) if has_med else None
    med_dec    = MedDecoder(z_dim=z_dim, med_dim=len(med_nn_idx)).to(DEVICE) \
                    if has_med else None

    X_t = torch.tensor(X_nn, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y_np, dtype=torch.float32).to(DEVICE)
    s_t = torch.tensor(s_train, dtype=torch.long).to(DEVICE)

    dataset = TensorDataset(X_t, y_t, s_t)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    params_main = (list(trunk.parameters()) +
                   list(split_head.parameters()) +
                   list(task_head.parameters()))
    opt_main = torch.optim.AdamW(params_main, lr=3e-4, weight_decay=1e-4)
    opt_sadv = torch.optim.AdamW(sens_adv.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_med  = torch.optim.AdamW(
        list(sens_probe.parameters()) + list(med_dec.parameters()),
        lr=3e-4, weight_decay=1e-4
    ) if has_med else None
    sch_main = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_main, T_max=total_ep, eta_min=1e-5)

    fw = FAIR_WEIGHTS.get(dataset_name, (5.0, 3.0, 1.0, 0.5, 1.0, 0.2))
    fw_fair, fw_sadv, fw_probe, fw_recon, fw_ortho, fw_gate = fw
    pred_std_min = PRED_STD_MIN.get(dataset_name, 0.02)
    ramp_centre  = RAMP_CENTRE.get(dataset_name, 0.08)
    ramp_steep   = RAMP_STEEP.get(dataset_name, 18.0)

    pareto  = ParetoCheckpoint(lam=lam)
    stopper = EarlyStopping(patience=patience, min_delta=1e-4)
    X_test  = data['X_test_nn']
    y_test  = data['y_test']

    for ep in range(1, total_ep + 1):
        trunk.train(); split_head.train(); task_head.train(); sens_adv.train()
        if sens_probe: sens_probe.train()
        if med_dec:    med_dec.train()

        fair = use_fair and (ep > warmup_ep)
        if fair:
            ramp_progress = (ep - warmup_ep) / max(1.0, total_ep - warmup_ep)
            fair_ramp = float(
                1.0 / (1.0 + math.exp(-ramp_steep * (ramp_progress - ramp_centre))))
        else:
            fair_ramp = 0.0
        alpha = min(2.0, fair_ramp * 2.0)

        for Xb, yb, sb in loader:
            h = trunk(Xb)
            z_task, z_med, gate = split_head(h)
            logit = task_head(z_task)

            bce  = F.binary_cross_entropy_with_logits(logit, yb)
            p_mean = torch.sigmoid(logit).mean()
            collapse_pen = 2.0 * (F.relu(0.10 - p_mean) + F.relu(p_mean - 0.90))
            loss = bce + collapse_pen
            loss = loss + fw_gate * split_head.gate_regularisation_loss(gate)

            if fair and fair_ramp > 0:
                fair_pen = fair_loss_dp_soft(logit, sb)
                loss = loss + fw_fair * fair_ramp * fair_pen

            if use_adv and fair and fair_ramp > 0:
                sadv_out = sens_adv(z_task, alpha=alpha)
                loss = loss + fw_sadv * fair_ramp * F.cross_entropy(sadv_out, sb)

            if fair and fair_ramp > 0 and has_med:
                loss = loss + fw_ortho * fair_ramp * _hsic(z_task, z_med)

            opt_main.zero_grad()
            opt_sadv.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trunk.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(split_head.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(task_head.parameters(), 1.0)
            opt_main.step()
            if use_adv: opt_sadv.step()

            if has_med and opt_med is not None:
                h2_base      = trunk(Xb).detach()
                h2_in        = h2_base.requires_grad_(True)
                _, z_med2, _ = split_head(h2_in)

                probe_out  = sens_probe(z_med2)
                loss_probe = F.cross_entropy(probe_out, sb)

                recon_out  = med_dec(z_med2)
                X_med_b    = Xb[:, med_nn_idx]
                loss_recon = F.mse_loss(recon_out, X_med_b)

                loss_med = fw_probe * loss_probe + fw_recon * loss_recon
                opt_med.zero_grad()
                loss_med.backward()
                opt_med.step()

        sch_main.step()

        if ep % 10 == 0 or ep == total_ep:
            proba    = get_proba(trunk, split_head, task_head, X_test, DEVICE)
            pred     = (proba > 0.5).astype(int)
            acc      = accuracy_score(y_test, pred)
            pred_std = float(np.std(proba))
            try:
                metric = abs(float(
                    demographic_parity_difference(
                        y_test, pred, sensitive_features=s_test)
                ))
            except Exception:
                metric = 1.0

            pos_rate = float(pred.mean())
            non_degenerate = pred_std >= pred_std_min and 0.05 <= pos_rate <= 0.95
            if non_degenerate:
                updated = pareto.update(
                    acc, metric,
                    trunk=trunk, split_head=split_head,
                    task_head=task_head, sens_probe=sens_probe)
                composite_loss = -(acc - lam * metric)
                if use_fair and stopper.update(composite_loss):
                    tqdm.write(f'  Early stop ep{ep}')
                    break

            if ep % 50 == 0:
                h_sample = trunk(X_t[:512])
                gate_mean, gate_std, gate_med_dims = \
                    split_head.get_gate_stats(h_sample)
                tqdm.write(f'  [DP] ep{ep}: acc={acc:.4f} '
                           f'dp={metric:.4f} std={pred_std:.3f} '
                           f'gate_med={gate_med_dims}/{z_dim} '
                           f'gate_mean={gate_mean:.3f}')

    pareto.restore(trunk, split_head, task_head, sens_probe)
    tqdm.write(f'  Restored Pareto best: dp={pareto.best_metric:.4f} '
               f'acc={pareto.best_acc:.4f} composite={pareto.best_composite:.4f}')

    proba  = get_proba(trunk, split_head, task_head, X_test, DEVICE)
    pred   = (proba > 0.5).astype(int)
    acc    = accuracy_score(y_test, pred)
    try:
        metric = abs(float(
            demographic_parity_difference(y_test, pred, sensitive_features=s_test)
        ))
    except Exception:
        metric = 1.0

    return acc, metric, trunk, split_head, task_head, sens_probe, med_dec


def run_fairlearn_expgrad(X_train, y_train, X_test, y_test, s_train, s_test):
    base = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED)
    constraint = DemographicParity()
    mitigator = ExponentiatedGradient(base, constraint, eps=0.01)
    try:
        mitigator.fit(X_train, y_train, sensitive_features=s_train)
        pred = mitigator.predict(X_test)
    except Exception as e:
        tqdm.write(f'  [ExpGrad] failed: {e}')
        lr = LogisticRegression(max_iter=1000, random_state=SEED)
        lr.fit(X_train, y_train)
        pred = lr.predict(X_test)
    acc = accuracy_score(y_test, pred)
    try:
        dp = abs(float(demographic_parity_difference(y_test, pred, sensitive_features=s_test)))
    except Exception:
        dp = 1.0
    bal = balanced_accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, zero_division=0)
    return acc, dp, bal, f1


def run_fairlearn_threshold(X_train, y_train, X_test, y_test, s_train, s_test):
    base = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=SEED)
    base.fit(X_train, y_train)
    optimizer = ThresholdOptimizer(
        estimator=base,
        constraints='demographic_parity',
        objective='accuracy_score',
        predict_method='predict_proba',
    )
    try:
        optimizer.fit(X_train, y_train, sensitive_features=s_train)
        pred = optimizer.predict(X_test, sensitive_features=s_test)
    except Exception as e:
        tqdm.write(f'  [ThreshOpt] failed: {e}')
        pred = base.predict(X_test)
    acc = accuracy_score(y_test, pred)
    try:
        dp = abs(float(demographic_parity_difference(y_test, pred, sensitive_features=s_test)))
    except Exception:
        dp = 1.0
    bal = balanced_accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, zero_division=0)
    return acc, dp, bal, f1


def compute_extended_metrics(y_true, pred, s):
    metrics = {}
    metrics['accuracy']          = accuracy_score(y_true, pred)
    metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, pred)
    metrics['f1']                = f1_score(y_true, pred, zero_division=0)
    try:
        metrics['dp_diff'] = abs(float(
            demographic_parity_difference(y_true, pred, sensitive_features=s)))
    except Exception:
        metrics['dp_diff'] = 1.0
    try:
        metrics['dp_ratio'] = float(
            demographic_parity_ratio(y_true, pred, sensitive_features=s))
    except Exception:
        metrics['dp_ratio'] = float('nan')
    return metrics


LOADERS = {
    'compas':   load_compas,
    'adult':    load_adult,
    'german':   load_german,
    'bank':     load_bank,
    'hospital': load_hospital,
}

print('=' * 70)
print(f'  LCFR — Causal Fair Representation Learning | {DEVICE}')
print('=' * 70)

all_results = {}

for idx, (dname, loader_fn) in enumerate(LOADERS.items(), 1):
    print(f'\n{"─" * 70}\n  [{idx}/{len(LOADERS)}] {dname.upper()}\n{"─" * 70}')
    set_seed()
    data     = loader_fn(use_cache=True)
    sens_cfg = DATASET_CONFIG[dname]['sens_attrs'][0]
    s_train  = np.asarray(data[sens_cfg[0]])
    s_test   = np.asarray(data[sens_cfg[1]])

    bbn_df    = data['bbn_train_df'].copy()
    bbn_df    = _enc_objects(bbn_df)
    label_col = 'y'
    if label_col not in bbn_df.columns:
        for c in bbn_df.columns:
            if c in ['two_year_recid', 'pass_bar', 'readmit_binary']:
                label_col = c
                break
    if label_col not in bbn_df.columns:
        bbn_df[label_col] = data['y_train'].astype(int)

    assert len(bbn_df) == len(s_train)
    bbn_df['_sens'] = s_train.astype(int)

    med_bbn, med_nn_idx, dag_info = learn_causal_mediators(
        bbn_df, '_sens', label_col, data['nn_feature_names'],
        alpha=PC_ALPHA.get(dname, 0.05),
        mi_alpha=MI_SCREEN_ALPHA.get(dname, 0.15),
        dataset_name=dname,
    )

    dset_results = {}

    print(f'\n  --- LCFR (full) ---')
    set_seed()
    acc, dp, trunk, sh, th, sp, md = train(
        dname, data, med_nn_idx, dag_info, s_train, s_test,
        use_adv=True, use_fair=True, use_dag_mask=True)
    proba = get_proba(trunk, sh, th, data['X_test_nn'], DEVICE)
    pred  = (proba > 0.5).astype(int)
    ext   = compute_extended_metrics(data['y_test'], pred, s_test)
    dset_results['LCFR'] = ext
    print(f'  LCFR: DP={ext["dp_diff"]:.4f}  Acc={ext["accuracy"]:.4f}  '
          f'BalAcc={ext["balanced_accuracy"]:.4f}  F1={ext["f1"]:.4f}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\n  --- Ablation: NoFair (no fairness objectives) ---')
    set_seed()
    acc_nf, dp_nf, trunk_nf, sh_nf, th_nf, _, _ = train(
        dname, data, med_nn_idx, dag_info, s_train, s_test,
        use_adv=False, use_fair=False, use_dag_mask=True)
    proba_nf = get_proba(trunk_nf, sh_nf, th_nf, data['X_test_nn'], DEVICE)
    pred_nf  = (proba_nf > 0.5).astype(int)
    ext_nf   = compute_extended_metrics(data['y_test'], pred_nf, s_test)
    dset_results['Abl-NoFair'] = ext_nf
    print(f'  NoFair: DP={ext_nf["dp_diff"]:.4f}  Acc={ext_nf["accuracy"]:.4f}  '
          f'BalAcc={ext_nf["balanced_accuracy"]:.4f}  F1={ext_nf["f1"]:.4f}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\n  --- Ablation: NoCausal (no DAG mask, no mediator HSIC) ---')
    set_seed()
    acc_nc, dp_nc, trunk_nc, sh_nc, th_nc, _, _ = train(
        dname, data, [], {}, s_train, s_test,
        use_adv=True, use_fair=True, use_dag_mask=False)
    proba_nc = get_proba(trunk_nc, sh_nc, th_nc, data['X_test_nn'], DEVICE)
    pred_nc  = (proba_nc > 0.5).astype(int)
    ext_nc   = compute_extended_metrics(data['y_test'], pred_nc, s_test)
    dset_results['Abl-NoCausal'] = ext_nc
    print(f'  NoCausal: DP={ext_nc["dp_diff"]:.4f}  Acc={ext_nc["accuracy"]:.4f}  '
          f'BalAcc={ext_nc["balanced_accuracy"]:.4f}  F1={ext_nc["f1"]:.4f}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\n  --- Ablation: NoAdv (no adversarial head) ---')
    set_seed()
    acc_na, dp_na, trunk_na, sh_na, th_na, _, _ = train(
        dname, data, med_nn_idx, dag_info, s_train, s_test,
        use_adv=False, use_fair=True, use_dag_mask=True)
    proba_na = get_proba(trunk_na, sh_na, th_na, data['X_test_nn'], DEVICE)
    pred_na  = (proba_na > 0.5).astype(int)
    ext_na   = compute_extended_metrics(data['y_test'], pred_na, s_test)
    dset_results['Abl-NoAdv'] = ext_na
    print(f'  NoAdv: DP={ext_na["dp_diff"]:.4f}  Acc={ext_na["accuracy"]:.4f}  '
          f'BalAcc={ext_na["balanced_accuracy"]:.4f}  F1={ext_na["f1"]:.4f}')
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f'\n  --- Baseline: ExpGrad (Fairlearn) ---')
    set_seed()
    acc_eg, dp_eg, bal_eg, f1_eg = run_fairlearn_expgrad(
        data['X_train_nn'], data['y_train'],
        data['X_test_nn'],  data['y_test'],
        s_train, s_test)
    dset_results['FL-ExpGrad'] = {
        'accuracy': acc_eg, 'dp_diff': dp_eg,
        'balanced_accuracy': bal_eg, 'f1': f1_eg}
    print(f'  ExpGrad: DP={dp_eg:.4f}  Acc={acc_eg:.4f}  '
          f'BalAcc={bal_eg:.4f}  F1={f1_eg:.4f}')

    print(f'\n  --- Baseline: ThresholdOptimizer (Fairlearn) ---')
    set_seed()
    acc_to, dp_to, bal_to, f1_to = run_fairlearn_threshold(
        data['X_train_nn'], data['y_train'],
        data['X_test_nn'],  data['y_test'],
        s_train, s_test)
    dset_results['FL-ThreshOpt'] = {
        'accuracy': acc_to, 'dp_diff': dp_to,
        'balanced_accuracy': bal_to, 'f1': f1_to}
    print(f'  ThreshOpt: DP={dp_to:.4f}  Acc={acc_to:.4f}  '
          f'BalAcc={bal_to:.4f}  F1={f1_to:.4f}')

    all_results[dname] = dset_results
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


METHODS = ['LCFR', 'Abl-NoFair', 'Abl-NoCausal', 'Abl-NoAdv',
           'FL-ExpGrad', 'FL-ThreshOpt']

print(f'\n{"=" * 70}')
print('  LCFR — MAIN RESULTS + ABLATIONS + BASELINES (Demographic Parity)')
print(f'{"=" * 70}')

header_fmt = f'  {"Method":<16} ' + ' '.join([f'{d.upper():>10}' for d in LOADERS])
subhdr_fmt = f'  {"":16} ' + ' '.join([f'{"DP/Acc":>10}'       for d in LOADERS])
print(header_fmt)
print(subhdr_fmt)
print(f'  {"─" * (16 + 12 * len(LOADERS))}')

for method in METHODS:
    row = f'  {method:<16} '
    for dname in LOADERS:
        r = all_results.get(dname, {}).get(method)
        if r:
            row += f'{r["dp_diff"]:.3f}/{r["accuracy"]:.3f} '
        else:
            row += f'{"—":>10} '
    print(row)

print(f'\n{"=" * 70}')
print('  Full metric table (Acc / BalAcc / F1 / DP)')
print(f'{"=" * 70}')

for dname in LOADERS:
    print(f'\n  {dname.upper()}')
    print(f'  {"Method":<16} {"Acc":>6} {"BalAcc":>8} {"F1":>6} {"DP":>8}')
    print(f'  {"─" * 44}')
    for method in METHODS:
        r = all_results.get(dname, {}).get(method)
        if r:
            print(f'  {method:<16} {r["accuracy"]:>6.4f} '
                  f'{r["balanced_accuracy"]:>8.4f} '
                  f'{r["f1"]:>6.4f} {r["dp_diff"]:>8.4f}')

print(f'\n{"=" * 70}')

  LCFR — Causal Fair Representation Learning | cuda

──────────────────────────────────────────────────────────────────────
  [1/5] COMPAS
──────────────────────────────────────────────────────────────────────
  [PC] compas: 8 nodes, n=4937, alpha=0.05
  [PC] Mediators (capped+leak guard): ['age', 'priors_count']
  [MI-screen] MI(S,Y)=0.0097  H(Y)=0.6891  thresh=0.0015
  [MI-screen] Candidates: ['priors_count', 'age']
  [Hybrid] Final mediators (2): ['age', 'priors_count']
    PC-only: []
    MI-only: []
    Both:    ['age', 'priors_count']
    NN idx:  [0, 1]/377

  --- LCFR (full) ---
  [DAG-mask] Boosted 8 dims to meet min_med_fraction=0.15
  [DAG-mask] med_score>0.5 dims: 9/64
  [DAG-mask] score dist: max=1.000 mean=0.103 min=0.000
  [DP] ep50: acc=0.6834 dp=0.2459 std=0.289 gate_med=9/64 gate_mean=0.139
  [DP] ep100: acc=0.6591 dp=0.0545 std=0.273 gate_med=9/64 gate_mean=0.140
  [DP] ep150: acc=0.6462 dp=0.0510 std=0.265 gate_med=9/64 gate_mean=0.140
  [DP] ep200: acc=0.6470 dp=0.